In [32]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV
)
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, recall_score, average_precision_score, confusion_matrix, f1_score
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks

In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"

sys.path.append(str(SRC_PATH))

In [35]:
# set global seaborn style
sns.set_theme(
    context="notebook",
    style="ticks",
    palette="muted",
    rc={
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

In [36]:
df = pd.read_csv("../data/processed/stroke_clean.csv")

In [37]:
df.head()

,gender,age,hypertension,heart_disease,ever_married,work_type,residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,male,67.0,0,1,1,private,urban,5.436731,3.627004,formerly_smoked,1
1,female,61.0,0,0,1,self-employed,rural,5.314240,NaN,never_smoked,1
2,male,80.0,0,1,1,private,rural,4.672081,3.511545,never_smoked,1
3,female,49.0,0,0,1,private,urban,5.148831,3.566712,smokes,1
4,female,79.0,1,0,1,self-employed,rural,5.165471,3.218876,never_smoked,1


In [38]:
X = df.drop(columns=["stroke"])
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [39]:
print("Train Stroke Rate:", y_train.mean())
print("Test Stroke Rate:", y_test.mean())

Train Stroke Rate: 0.04869097137264497
Test Stroke Rate: 0.04892367906066536


In [40]:
continuous_vars = [
    "age",
    "avg_glucose_level",
    "bmi"
]

binary_vars = [
    "hypertension",
    "heart_disease",
    "ever_married"
]

categorical_vars = [
    "gender",
    "work_type",
    "residence_type",
    "smoking_status",
]

In [41]:
from preprocessing import build_preprocessor_median, build_preprocessor_knn

preprocessor_median = build_preprocessor_median(
    continuous_vars,
    categorical_vars,
    binary_vars,
)

preprocessor_knn = build_preprocessor_knn(
    continuous_vars,
    categorical_vars,
    binary_vars,
)

In [ ]:
pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    "XGBoost": XGBClassifier(
        n_jobs=-1,
        scale_pos_weight=pos_weight,
        random_state=42,
        eval_metric="logloss",
    ),
}

19.537688442211056


In [44]:
def evaluate_model(model, X_test, y_test) -> dict:
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)

    return {
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

In [45]:
results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ("preprocessor", preprocessor_median),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(pipeline, X_test, y_test)
    results[name] = metrics

    print(f"\n=== {name} ===")
    for k, v in metrics.items():
        print(k, ":\n", v)


=== RandomForest ===
roc_auc :
 0.7943930041152263
pr_auc :
 0.15419373585970014
recall :
 0.0
f1 :
 0.0
confusion_matrix :
 [[971   1]
 [ 50   0]]

=== XGBoost ===
roc_auc :
 0.7580658436213992
pr_auc :
 0.14145936592105807
recall :
 0.16
f1 :
 0.15841584158415842
confusion_matrix :
 [[929  43]
 [ 42   8]]


In [48]:
pipeline_xgb_base = ImbPipeline([
    ("preprocessor", preprocessor_knn),
    ("model", XGBClassifier(
        n_jobs=-1,
        scale_pos_weight=pos_weight,
        random_state=42,
        eval_metric="logloss",
    ))
])

pipeline_xgb_smote = ImbPipeline([
    ("preprocessor", preprocessor_knn),
    ("smote", SMOTE(sampling_strategy="minority", random_state=42)),
    ("tomek", TomekLinks(sampling_strategy="majority")),
    ("model", XGBClassifier(
        n_jobs=-1,
        random_state=42,
        eval_metric="logloss",
    ))
])

In [58]:
params_xgb = {
    "model__max_depth": [3, 4, 5, 6],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__n_estimators": [100, 200, 300],
    "model__gamma": [0, 0.1, 0.2],
    "model__min_child_weight": [1, 3, 5],
    "model__subsample": [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.6, 0.8, 1.0]
}

In [59]:
search_base = RandomizedSearchCV(
    estimator=pipeline_xgb_base,
    param_distributions=params_xgb,
    n_iter=20,
    scoring="average_precision",
    n_jobs=-1,
    cv=5,
    verbose=1,
    random_state=42,
)

search_base.fit(X_train, y_train)

best_base = search_base.best_estimator_
print("Best Hyperparameters:", search_base.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Hyperparameters: {'model__subsample': 0.7, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__gamma': 0.1, 'model__colsample_bytree': 0.6}


In [60]:
search_smote = RandomizedSearchCV(
    estimator=pipeline_xgb_smote,
    param_distributions=params_xgb,
    n_iter=20,
    scoring="average_precision",
    n_jobs=-1,
    cv=5,
    verbose=1,
    random_state=42,
)

search_smote.fit(X_train, y_train)

best_smote = search_smote.best_estimator_
print("Best Hyperparameters:", search_smote.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Hyperparameters: {'model__subsample': 0.7, 'model__n_estimators': 300, 'model__min_child_weight': 1, 'model__max_depth': 3, 'model__learning_rate': 0.01, 'model__gamma': 0.1, 'model__colsample_bytree': 0.6}


In [61]:
results_base = evaluate_model(best_base, X_test, y_test)
results_smote = evaluate_model(best_smote, X_test, y_test)

print("\n=== XGB + scale_pos_weight ===")
for k, v in results_base.items():
    print(k, ":\n", v)

print("\n=== XGB + SMOTE ===")
for k, v in results_smote.items():
    print(k, ":\n", v)


=== XGB + scale_pos_weight ===
roc_auc :
 0.8441769547325103
pr_auc :
 0.224289269041144
recall :
 0.82
f1 :
 0.21298701298701297
confusion_matrix :
 [[678 294]
 [  9  41]]

=== XGB + SMOTE ===
roc_auc :
 0.8188888888888888
pr_auc :
 0.19988409066507437
recall :
 0.76
f1 :
 0.22157434402332363
confusion_matrix :
 [[717 255]
 [ 12  38]]
